# 北上广深租房市场数据分析

## 第一部分：数据读取与初步检查

数据来源：Alfred1984/interesting-python 项目的 BSGS_Rent 样本数据。

In [8]:
Path.cwd()

WindowsPath('C:/Users/ASUS/Desktop/Data Analysis/China-rent-analysis/notebooks')

In [9]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/data_sample.csv")

df = pd.read_csv(DATA_PATH)

print("数据规模：", df.shape)
print("总行数：", df.shape[0])
print("总字段数：", df.shape[1])

df.head()

数据规模： (12000, 20)
总行数： 12000
总字段数： 20


,_id,bathroom_num,bedroom_num,bizcircle_name,city,dist,distance,frame_orientation,hall_num,house_tag,house_title,latitude,layout,longitude,m_url,rent_area,rent_price_listing,rent_price_unit,resblock_name,type
0,5c714363397be4c5251a3ded,2,3,上地,北京,海淀,NaN,南 北,2,精装 集中供暖 双卫生间,整租 · 上地西里二层三居 自住出租 随时入住采光好 无遮挡,40.039000,3室2厅2卫,116.317831,https://m.lianjia.com/chuzu/bj/zufang/BJ213593...,137,15000,元/月,上地西里,整租
1,5c7148e6397be4c5251a583d,1,2,北大地,北京,丰台,NaN,南 北,1,集中供暖,整租 · 丰台北大地电报局街家具家电齐全南北向两居,39.856662,2室1厅1卫,116.292291,https://m.lianjia.com/chuzu/bj/zufang/BJ210122...,57,4500,元/月,电报局街,整租
2,5c71321e397be4c5251a0b46,1,1,燕莎,北京,朝阳,788.0,北,1,近地铁 精装 集中供暖,整租 · 远洋新干线 1室1厅 10500元,39.963231,1室1厅1卫,116.466150,https://m.lianjia.com/chuzu/bj/zufang/BJ216317...,56,10500,元/月,远洋新干线,整租
3,5c712721397be4c52519fc09,1,1,阜成门,北京,西城,886.0,东,1,近地铁 集中供暖 随时看房,南露园 1室1厅 5600元,39.930655,1室1厅1卫,116.348521,https://m.lianjia.com/chuzu/bj/zufang/BJ217362...,43,5600,元/月,南露园,整租
4,5c7123bd397be4c52519f7af,1,2,和平里,北京,朝阳,779.0,东 南,1,近地铁 集中供暖,和平里东街15号院 2室1厅 6300元,39.957079,2室1厅1卫,116.431393,https://m.lianjia.com/chuzu/bj/zufang/BJ215869...,56,6300,元/月,和平里东街15号院,整租


In [10]:
df.info() # 查看每个字段的数据类型和缺失情况

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 12000 non-null  str    
 1   bathroom_num        12000 non-null  int64  
 2   bedroom_num         12000 non-null  int64  
 3   bizcircle_name      11999 non-null  str    
 4   city                12000 non-null  str    
 5   dist                12000 non-null  str    
 6   distance            6794 non-null   float64
 7   frame_orientation   11899 non-null  str    
 8   hall_num            12000 non-null  int64  
 9   house_tag           10124 non-null  str    
 10  house_title         12000 non-null  str    
 11  latitude            11969 non-null  float64
 12  layout              12000 non-null  str    
 13  longitude           11969 non-null  float64
 14  m_url               12000 non-null  str    
 15  rent_area           12000 non-null  str    
 16  rent_price_list

In [11]:
# 显示存在缺失值的字段，并从缺失最严重到最轻微排序
missing_count = df.isna().sum()
missing_rate = (missing_count / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失比例(%)": missing_rate
})

missing_report = missing_report[
    missing_report["缺失数量"] > 0
].sort_values("缺失比例(%)", ascending=False)

missing_report

,缺失数量,缺失比例(%)
distance,5206,43.38
house_tag,1876,15.63
resblock_name,1514,12.62
frame_orientation,101,0.84
latitude,31,0.26
longitude,31,0.26
bizcircle_name,1,0.01


In [12]:
# 检查重复数据
duplicate_rows = df.duplicated().sum()
duplicate_ids = df["_id"].duplicated().sum()
unique_ids = df["_id"].nunique()

print("完全重复的记录数：", duplicate_rows)
print("重复的房源ID数量：", duplicate_ids)
print("唯一房源ID数量：", unique_ids)

完全重复的记录数： 0
重复的房源ID数量： 0
唯一房源ID数量： 12000


In [13]:
# 查看与租金分析最相关的字段
key_columns = [
    "city",
    "dist",
    "bizcircle_name",
    "type",
    "layout",
    "rent_area",
    "rent_price_listing",
    "rent_price_unit"
]

df[key_columns].head(10)

,city,dist,bizcircle_name,type,layout,rent_area,rent_price_listing,rent_price_unit
0,北京,海淀,上地,整租,3室2厅2卫,137,15000,元/月
1,北京,丰台,北大地,整租,2室1厅1卫,57,4500,元/月
2,北京,朝阳,燕莎,整租,1室1厅1卫,56,10500,元/月
3,北京,西城,阜成门,整租,1室1厅1卫,43,5600,元/月
4,北京,朝阳,和平里,整租,2室1厅1卫,56,6300,元/月
5,北京,朝阳,奥林匹克公园,整租,2室2厅1卫,111,12000,元/月
6,北京,顺义,顺义城,整租,2室1厅1卫,65,3600,元/月
7,北京,朝阳,农展馆,整租,3室2厅3卫,272,32000,元/月
8,北京,通州,玉桥,整租,2室1厅1卫,89,5000,元/月
9,北京,亦庄开发区,亦庄,整租,2房间1卫,49,5200,元/月


In [14]:
# 检查关键分类字段有哪些取值
print("城市分布：")
display(df["city"].value_counts(dropna=False))

print("出租类型分布：")
display(df["type"].value_counts(dropna=False))

print("租金单位分布：")
display(df["rent_price_unit"].value_counts(dropna=False))

城市分布：


city
北京    3000
上海    3000
广州    3000
深圳    3000
Name: count, dtype: int64

出租类型分布：


type
整租    11586
合租      414
Name: count, dtype: int64

租金单位分布：


rent_price_unit
元/月    12000
Name: count, dtype: int64

In [15]:
# 检查面积和租金能否全部转换成数值(有些数值是区间)
area_numeric = pd.to_numeric(
    df["rent_area"],
    errors="coerce"
)

price_numeric = pd.to_numeric(
    df["rent_price_listing"],
    errors="coerce"  # 遇到不能转换成数字的内容，先标记成缺失值，而不是让程序报错
)

print("无法转换为数值的面积数量：", area_numeric.isna().sum())
print("无法转换为数值的租金数量：", price_numeric.isna().sum())

print("\n异常面积原始值：")
display(
    df.loc[area_numeric.isna(), "rent_area"]
      .value_counts()
      .head(20)
)

print("\n异常租金原始值：")
display(
    df.loc[price_numeric.isna(), "rent_price_listing"]
      .value_counts()
      .head(20)
)

无法转换为数值的面积数量： 250
无法转换为数值的租金数量： 524

异常面积原始值：


rent_area
30-35    15
20-25    11
18-20     7
40-45     6
25-28     6
25-30     5
28-33     4
30-40     4
15-18     4
15-20     4
23-25     4
30-32     3
20-30     3
15-25     3
45-50     3
30-45     3
15-16     3
20-22     3
26-30     3
16-18     3
Name: count, dtype: int64


异常租金原始值：


rent_price_listing
1400-1500    7
1200-1300    5
850-900      5
900-1000     5
1300-1400    5
1800-2000    5
1580-1680    4
800-850      4
1000-1100    4
1500-1600    4
800-1000     4
700-800      4
900-1200     4
700-750      4
1600-1700    4
1350-1400    4
750-800      3
1300-1500    3
1680-1880    3
600-700      3
Name: count, dtype: int64

# 数据概况与初步发现

### 数据概况

- 数据包含 12,000 条房源记录和 20 个字段。
- 北京、上海、广州、深圳各有 3,000 条记录，城市样本数量均衡。
- 整租房源 11,586 条，合租房源 414 条。
- 所有房源的租金单位均为“元/月”。
- 房源 ID 没有重复，每条记录代表一个独立房源。

### 数据质量问题

- `distance` 缺失 5,206 条，缺失比例为 43.38%。
- `house_tag` 缺失 1,876 条，缺失比例为 15.63%。
- `resblock_name` 缺失 1,514 条，缺失比例为 12.62%。
- 面积字段中有 250 条区间数据，例如 `30-35`。
- 租金字段中有 524 条区间数据，例如 `1400-1500`。
- 面积和租金字段需要在清洗阶段转换为统一数值。

### 待分析的业务问题

1. 北上广深哪个城市的租金中位数最高？
2. 哪些区域的单位面积租金最高？
3. 房屋面积与月租金之间有什么关系？
4. 不同房型和出租类型的租金差异有多大？
5. 在给定预算下，哪个城市或区域的可选房源最多？